# Cool Roof Priority Index — Mussafah, Abu Dhabi

**Arab Youth Space Hackathon 2026 · 813 Challenge**
**Theme:** Urban Expansion, Land Use Change & Heat Risk · **Track:** Hyperspectral

---

### What this notebook does

A municipality knows its city is hot. What it does not know is **which specific
roof to treat first**, and **how much cooling that treatment buys**. This notebook
turns open Earth observation into a ranked, costed worklist of individual rooftops.

Five steps, each one checkable:

| # | Step | Data | Output |
|---|------|------|--------|
| 1 | Measure heat | Landsat C2 L2 Surface Temperature, 2 epochs | LST maps, ΔLST |
| 2 | Measure surface brightness | Sentinel-2 L2A → broadband albedo | albedo map |
| 3 | Map urban expansion | built-up classification, 2 epochs | hectares converted, validated vs ESA WorldCover |
| 4 | **Calibrate the lever** | regress LST on albedo over built-up pixels | **°C of cooling per unit albedo, locally measured** |
| 5 | Rank the roofs | OSM footprints + zonal statistics | ranked CSV of roofs by cooling gained |

### Why step 4 is the point

Everyone can draw a heat map. The question a planner actually asks is *"if I spend
money here, how much cooler does it get?"* We answer it by **measuring the
albedo–temperature relationship in this city, from this city's own data**, instead
of importing a coefficient from a paper about Los Angeles. That makes the
intervention estimate locally calibrated and falsifiable.

### Honest framing

Land surface temperature is not air temperature. Our outputs are a **screening and
prioritisation indicator**, not a thermal model of the built environment, and not
ground truth. Section 9 states precisely where this breaks.

**Runtime:** ~25–40 minutes on a free Colab CPU instance. No GPU needed.


## 1 · Environment

Pinned versions. If a cell fails after a Colab restart, re-run this one first.

In [ ]:
%pip install -q \
    pystac-client==0.8.6 \
    planetary-computer==1.0.0 \
    stackstac==0.5.1 \
    rioxarray==0.17.0 \
    xarray==2024.7.0 \
    rasterio==1.3.11 \
    geopandas==1.0.1 \
    shapely==2.0.6 \
    rasterstats==0.20.0 \
    scikit-learn==1.5.2 \
    matplotlib==3.9.2 \
    pandas==2.2.3 \
    requests==2.32.4

In [ ]:
# ---------------------------------------------------------------------------
# Bootstrap. Works in three situations, so you never have to think about paths:
#   1. Colab, notebook uploaded on its own  -> clones the repo and moves into it
#   2. Colab, repo already cloned           -> just moves into notebooks/
#   3. Local jupyter inside the repo        -> does nothing
# Set REPO_URL to your own repo once you have created it.
# ---------------------------------------------------------------------------
import os, sys, subprocess

REPO_URL  = "https://github.com/RISKYXBT/coolroof-mussafah.git"
REPO_DIR  = "coolroof-mussafah"

def _in_repo():
    return os.path.isdir("../results") and os.path.isdir("../data")

if not _in_repo():
    if os.path.isdir(f"/content/{REPO_DIR}/notebooks"):
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    elif "google.colab" in sys.modules:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            print(f"Cloning {REPO_URL} ...")
            r = subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print("Clone failed:", r.stderr.strip())
                print("Falling back to a local working folder.")
                os.makedirs(f"{REPO_DIR}/notebooks", exist_ok=True)
                os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
                os.makedirs(f"{REPO_DIR}/data/sample_input", exist_ok=True)
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    else:
        # Local, but not inside the repo: create the folders beside us.
        os.makedirs("../results", exist_ok=True)
        os.makedirs("../data/sample_input", exist_ok=True)

print("working directory :", os.getcwd())
print("results folder    :", os.path.abspath("../results"))


In [ ]:
import os, json, warnings, requests
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import pystac_client, planetary_computer, stackstac
from shapely.geometry import box, shape, Polygon
from rasterstats import zonal_stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")
np.random.seed(42)           # reproducible fold assignment
plt.rcParams["figure.dpi"] = 110

# Paths are RELATIVE so a reviewer can clone and run without editing anything.
RESULTS = "../results"
SAMPLE  = "../data/sample_input"
for d in (RESULTS, SAMPLE):
    os.makedirs(d, exist_ok=True)

print("environment ready")

## 2 · Configuration

Every parameter the analysis depends on lives in this one cell, so a reviewer can
see the whole experimental setup at a glance and change the city in one place.

In [ ]:
# ---- Area of interest -------------------------------------------------------
# Mussafah, Abu Dhabi: the emirate's main industrial and light-manufacturing
# district. Chosen because it has (a) very large flat roofs, which are the
# cheapest cool-roof retrofit targets per square metre, (b) a strong thermal
# signal, and (c) a decade of visible expansion into bare desert, which gives us
# the land-use-change half of the theme.
AOI_NAME  = "Mussafah, Abu Dhabi"
AOI_BBOX  = [54.43, 24.29, 54.58, 24.41]        # [W, S, E, N]

# Smaller core used for the per-building step, so the OSM query and the zonal
# statistics stay fast. Start small, as the onboarding guide instructs.
FOCUS_BBOX = [54.48, 24.33, 54.55, 24.38]

# ---- Time windows -----------------------------------------------------------
# UAE peak heat is June-August. We widen to May-September and span three years
# per epoch so enough cloud-free scenes accumulate for a stable median.
EPOCH_A      = "2014-05-01/2016-09-30"      # "before"
EPOCH_B      = "2024-05-01/2026-09-30"      # "after"
EPOCH_A_NAME = "2014-2016"
EPOCH_B_NAME = "2024-2026"

MAX_CLOUD = 15          # percent, scene level
EPSG      = 32640       # UTM 40N - correct zone for Abu Dhabi
RES_LST   = 30          # m, Landsat native
RES_S2    = 20          # m, so B11/B12 are not upsampled

# ---- Intervention assumption ------------------------------------------------
# Albedo a commercial cool-roof coating typically reaches. The cooling estimate
# is driven by the measured local sensitivity (Section 7), NOT by this number -
# this only sets how far each roof could move.
TARGET_ALBEDO = 0.60

print(f"AOI   : {AOI_NAME}  {AOI_BBOX}")
print(f"focus : {FOCUS_BBOX}")
print(f"epochs: {EPOCH_A_NAME} vs {EPOCH_B_NAME}")

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("connected to Planetary Computer STAC")

## 3 · Land surface temperature from Landsat Collection 2 Level-2

**A deliberate choice worth defending in the pitch.** We do *not* convert
top-of-atmosphere radiance to temperature ourselves. USGS Collection 2 **Level-2**
already ships an atmospherically corrected Surface Temperature product (`lwir11`,
derived from band 10 with an emissivity model from ASTER GED). Using it means:

- fewer hand-made assumptions for a judge to question,
- a documented, citable processing chain,
- and our effort goes into the part that is actually novel.

Scaling (from the USGS Collection 2 Level-2 spec):
`Kelvin = DN × 0.00341802 + 149.0`

Clouds are masked per-pixel with `qa_pixel` bit flags before compositing, because
a single undetected cloud drags a median down by several degrees.

In [ ]:
# QA_PIXEL bit flags, Landsat Collection 2
QA_FILL, QA_DILATED, QA_CIRRUS, QA_CLOUD, QA_SHADOW = 0, 1, 2, 3, 4

def landsat_lst(bbox, window, label):
    """Cloud-masked median summer Land Surface Temperature in degrees Celsius."""
    items = list(catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=window,
        query={"eo:cloud_cover": {"lt": MAX_CLOUD},
               "platform": {"in": ["landsat-8", "landsat-9"]}},
    ).items())
    if not items:
        raise RuntimeError(f"No Landsat scenes for {label}. Widen the window "
                           f"or raise MAX_CLOUD.")
    print(f"{label}: {len(items)} scenes "
          f"({min(i.properties['datetime'][:10] for i in items)} -> "
          f"{max(i.properties['datetime'][:10] for i in items)})")

    stack = stackstac.stack(
        items,
        assets=["lwir11", "qa_pixel"],
        resolution=RES_LST,
        bounds_latlon=bbox,
        epsg=EPSG,
        rescale=False,            # we apply the scale factor explicitly, below
        dtype="float32",
        fill_value=np.nan,
        chunksize=2048,
    )

    qa = stack.sel(band="qa_pixel").astype("uint16")
    bad = ((qa >> QA_FILL) & 1) | ((qa >> QA_DILATED) & 1) | \
          ((qa >> QA_CIRRUS) & 1) | ((qa >> QA_CLOUD) & 1) | \
          ((qa >> QA_SHADOW) & 1)

    st = stack.sel(band="lwir11")
    st = st.where(st > 0)                      # 0 is fill in the ST product
    kelvin = st * 0.00341802 + 149.0
    celsius = (kelvin - 273.15).where(bad == 0)

    lst = celsius.median(dim="time", skipna=True).compute()
    lst.name = f"LST_{label}"
    n_valid = int(np.isfinite(lst.values).sum())
    print(f"   median composite: {lst.shape}, {n_valid:,} valid pixels, "
          f"range {np.nanmin(lst.values):.1f} to {np.nanmax(lst.values):.1f} C")
    return lst, items

lst_a, items_a = landsat_lst(AOI_BBOX, EPOCH_A, EPOCH_A_NAME)
lst_b, items_b = landsat_lst(AOI_BBOX, EPOCH_B, EPOCH_B_NAME)

In [ ]:
# Align epoch A onto epoch B's grid, then difference.
lst_a_on_b = lst_a.interp_like(lst_b, method="nearest")
d_lst = (lst_b - lst_a_on_b).rename("delta_LST")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
vmin, vmax = 35, 60

for ax, arr, title in zip(
    axes[:2], [lst_a_on_b, lst_b],
    [f"Summer LST {EPOCH_A_NAME}", f"Summer LST {EPOCH_B_NAME}"]
):
    im = ax.imshow(arr.values, cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, label="degrees C")

lim = float(np.nanpercentile(np.abs(d_lst.values), 98))
im = axes[2].imshow(d_lst.values, cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[2].set_title(f"Change in LST ({EPOCH_A_NAME} to {EPOCH_B_NAME})"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046, label="degrees C")

plt.suptitle(f"Summer median land surface temperature - {AOI_NAME}",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS}/01_lst_change.png", bbox_inches="tight")
plt.show()

print(f"Mean LST {EPOCH_A_NAME}: {np.nanmean(lst_a_on_b.values):.2f} C")
print(f"Mean LST {EPOCH_B_NAME}: {np.nanmean(lst_b.values):.2f} C")
print(f"Mean change            : {np.nanmean(d_lst.values):+.2f} C")

## 4 · Surface albedo and land-cover indices from Sentinel-2

Albedo is the physical quantity a cool roof actually changes, so it has to be
measured, not assumed. We convert Sentinel-2 narrow bands to broadband shortwave
albedo using published narrow-to-broadband coefficients:

```
α = 0.2266·B2 + 0.1236·B3 + 0.1573·B4 + 0.3417·B8 + 0.1170·B11 + 0.0338·B12
```

The weights sum to 1.0.

> **Team to verify before submission:** confirm and cite the source for these
> coefficients (Bonafoni & Sekertekin, *Albedo Retrieval From Sentinel-2 by New
> Narrow-to-Broadband Conversion Coefficients*, IEEE GRSL 2020). The README must
> name the paper. Do not ship an uncited formula — *validity* is a judged
> criterion and an unsourced constant is the first thing a reviewer will poke.

In [ ]:
S2_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
ALBEDO_W = dict(B02=0.2266, B03=0.1236, B04=0.1573, B08=0.3417, B11=0.1170, B12=0.0338)
SCL_BAD  = [0, 1, 3, 8, 9, 10, 11]      # nodata, saturated, shadow, clouds, cirrus, snow

def sentinel2_composite(bbox, window, label):
    items = list(catalog.search(
        collections=["sentinel-2-l2a"], bbox=bbox, datetime=window,
        query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    ).items())
    if not items:
        raise RuntimeError(f"No Sentinel-2 scenes for {label}.")
    print(f"{label}: {len(items)} Sentinel-2 scenes")

    stack = stackstac.stack(
        items, assets=S2_BANDS + ["SCL"], resolution=RES_S2,
        bounds_latlon=bbox, epsg=EPSG, rescale=False,
        dtype="float32", fill_value=np.nan, chunksize=2048,
    )
    scl = stack.sel(band="SCL")
    good = ~scl.isin(SCL_BAD)
    refl = stack.sel(band=S2_BANDS).where(good) / 10000.0   # L2A scale factor
    comp = refl.median(dim="time", skipna=True).compute()
    print(f"   composite shape {comp.shape}")
    return comp

s2 = sentinel2_composite(AOI_BBOX, EPOCH_B, EPOCH_B_NAME)

b = {name: s2.sel(band=name) for name in S2_BANDS}
eps = 1e-6

albedo = sum(w * b[n] for n, w in ALBEDO_W.items()).rename("albedo")
ndvi   = ((b["B08"] - b["B04"]) / (b["B08"] + b["B04"] + eps)).rename("ndvi")
ndbi   = ((b["B11"] - b["B08"]) / (b["B11"] + b["B08"] + eps)).rename("ndbi")
mndwi  = ((b["B03"] - b["B11"]) / (b["B03"] + b["B11"] + eps)).rename("mndwi")

print(f"albedo range: {np.nanpercentile(albedo.values,1):.3f} "
      f"to {np.nanpercentile(albedo.values,99):.3f} "
      f"(median {np.nanmedian(albedo.values):.3f})")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
specs = [
    (albedo, "Broadband albedo", "gray",     0.05, 0.55),
    (ndvi,   "NDVI (vegetation)", "RdYlGn", -0.1,  0.6),
    (ndbi,   "NDBI (built-up)",   "RdYlBu_r", -0.3, 0.5),
    (mndwi,  "MNDWI (water)",     "Blues",   -0.6,  0.4),
]
for ax, (arr, title, cmap, lo, hi) in zip(axes, specs):
    im = ax.imshow(arr.values, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(title); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(f"Sentinel-2 surface properties, {EPOCH_B_NAME} summer median",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS}/02_s2_surface.png", bbox_inches="tight")
plt.show()

## 5 · Built-up classification, and validating it against an independent map

We classify with simple, explainable index thresholds — the onboarding guide is
explicit that a simple baseline comes before anything complex, and a threshold
rule is one a municipal GIS officer can audit.

Then we do the part most teams skip: **check it against ESA WorldCover**, an
independently produced 10 m land-cover product, and report a confusion matrix with
accuracy, precision, recall, F1 and IoU. This is what the organizers ask for when
they say *"not just 'it looks right on the map'"*.

In [ ]:
# Simple, auditable decision rule. Order matters: water, then vegetation,
# then built-up, then bare.
def classify(ndvi_a, ndbi_a, mndwi_a):
    lc = np.full(ndvi_a.shape, 3, dtype=np.uint8)      # 3 = bare / desert
    lc[np.asarray(ndbi_a) > 0.0] = 2                   # 2 = built-up
    lc[np.asarray(ndvi_a) > 0.20] = 1                  # 1 = vegetation
    lc[np.asarray(mndwi_a) > 0.10] = 0                 # 0 = water
    return lc

lc_b = classify(ndvi.values, ndbi.values, mndwi.values)
LABELS = ["Water", "Vegetation", "Built-up", "Bare / desert"]

counts = {LABELS[i]: int((lc_b == i).sum()) for i in range(4)}
total = sum(counts.values())
print(f"Classified {total:,} pixels at {RES_S2} m:")
for k, v in counts.items():
    print(f"   {k:16s} {v:9,}  ({v/total*100:5.1f}%)   {v*RES_S2**2/1e6:8.1f} km2")

In [ ]:
# ---- Independent reference: ESA WorldCover -----------------------------------
wc_items = list(catalog.search(collections=["esa-worldcover"], bbox=AOI_BBOX).items())
print(f"ESA WorldCover items found: {len(wc_items)}")

wc = stackstac.stack(
    wc_items, assets=["map"], resolution=RES_S2, bounds_latlon=AOI_BBOX,
    epsg=EPSG, rescale=False, dtype="float32", fill_value=np.nan,
).isel(band=0).max(dim="time").compute()

wc_on_grid = wc.interp_like(ndvi, method="nearest").values
WC_BUILTUP = 50          # ESA WorldCover class code for Built-up

ref_builtup = (wc_on_grid == WC_BUILTUP)
our_builtup = (lc_b == 2)
valid = np.isfinite(wc_on_grid) & np.isfinite(ndvi.values)

ref = ref_builtup[valid]; ours = our_builtup[valid]

TP = int(( ours &  ref).sum()); FP = int(( ours & ~ref).sum())
FN = int((~ours &  ref).sum()); TN = int((~ours & ~ref).sum())

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if TP + FP else float("nan")
recall    = TP / (TP + FN) if TP + FN else float("nan")
f1        = 2 * precision * recall / (precision + recall) if precision + recall else float("nan")
iou       = TP / (TP + FP + FN) if TP + FP + FN else float("nan")

print("\nBuilt-up classification vs ESA WorldCover 2021 (reference)")
print(f"   pixels compared : {valid.sum():,}")
print(f"   confusion       : TP={TP:,}  FP={FP:,}  FN={FN:,}  TN={TN:,}")
print(f"   accuracy        : {accuracy:.3f}")
print(f"   precision       : {precision:.3f}")
print(f"   recall          : {recall:.3f}")
print(f"   F1              : {f1:.3f}")
print(f"   IoU             : {iou:.3f}")

validation_builtup = dict(n=int(valid.sum()), TP=TP, FP=FP, FN=FN, TN=TN,
                          accuracy=accuracy, precision=precision,
                          recall=recall, f1=f1, iou=iou)

print("\nNOTE for the README: WorldCover is itself a model output, not ground "
      "truth. Agreement here means our map is consistent with an independent "
      "published product - it is not a field-verified accuracy figure.")

## 6 · Urban expansion, and what it cost in degrees

The theme asks us to quantify urban growth *and* link it to heat. Here we classify
both epochs, measure how much bare desert became built-up, and then compare the
temperature of those converted pixels against land that did not change.

This is the headline number for the pitch: **what one hectare of desert-to-industrial
conversion did to local surface temperature.**

In [ ]:
s2_a = sentinel2_composite(AOI_BBOX, EPOCH_A, EPOCH_A_NAME)
ba = {n: s2_a.sel(band=n) for n in S2_BANDS}
ndvi_a  = ((ba["B08"] - ba["B04"]) / (ba["B08"] + ba["B04"] + eps))
ndbi_a  = ((ba["B11"] - ba["B08"]) / (ba["B11"] + ba["B08"] + eps))
mndwi_a = ((ba["B03"] - ba["B11"]) / (ba["B03"] + ba["B11"] + eps))

lc_a = classify(ndvi_a.interp_like(ndvi).values,
                ndbi_a.interp_like(ndvi).values,
                mndwi_a.interp_like(ndvi).values)

px_km2 = RES_S2 ** 2 / 1e6
new_builtup    = (lc_a == 3) & (lc_b == 2)     # desert -> built
stable_builtup = (lc_a == 2) & (lc_b == 2)
stable_desert  = (lc_a == 3) & (lc_b == 3)

print(f"Desert converted to built-up : {new_builtup.sum()*px_km2:8.2f} km2 "
      f"({new_builtup.sum()*px_km2*100:,.0f} hectares)")
print(f"Built-up in both epochs      : {stable_builtup.sum()*px_km2:8.2f} km2")
print(f"Desert in both epochs        : {stable_desert.sum()*px_km2:8.2f} km2")

In [ ]:
# Put LST on the Sentinel-2 grid so the masks line up.
lst_b_s2 = lst_b.interp_like(ndvi, method="linear").values
lst_a_s2 = lst_a_on_b.interp_like(ndvi, method="linear").values
d_lst_s2 = lst_b_s2 - lst_a_s2

rows = []
for name, mask in [("New built-up (was desert)", new_builtup),
                   ("Stable built-up", stable_builtup),
                   ("Stable desert", stable_desert)]:
    m = mask & np.isfinite(d_lst_s2)
    if m.sum() < 50:
        continue
    rows.append(dict(
        category=name, n_pixels=int(m.sum()), area_km2=round(m.sum()*px_km2, 2),
        LST_before=round(float(np.nanmean(lst_a_s2[m])), 2),
        LST_after=round(float(np.nanmean(lst_b_s2[m])), 2),
        delta_LST=round(float(np.nanmean(d_lst_s2[m])), 2),
        mean_albedo=round(float(np.nanmean(albedo.values[m])), 3),
    ))

transition = pd.DataFrame(rows)
transition.to_csv(f"{RESULTS}/transition_summary.csv", index=False)
display(transition)

if len(transition) >= 2:
    try:
        conv = transition.loc[transition.category.str.startswith("New"), "delta_LST"].iloc[0]
        des  = transition.loc[transition.category.str.startswith("Stable desert"), "delta_LST"].iloc[0]
        print(f"\nHEADLINE: land converted from desert to built-up warmed "
              f"{conv - des:+.2f} C more than desert that was left alone.")
        print("This isolates the urbanisation signal from the regional warming "
              "trend, because both categories experienced the same climate.")
    except (IndexError, KeyError):
        pass

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
change = np.zeros_like(lc_b, dtype=np.uint8)
change[stable_desert]  = 1
change[stable_builtup] = 2
change[new_builtup]    = 3
cmap = ListedColormap(["#ffffff", "#f0e0c0", "#9e9e9e", "#d32f2f"])
axes[0].imshow(change, cmap=cmap, vmin=0, vmax=3); axes[0].axis("off")
axes[0].set_title(f"Urban expansion {EPOCH_A_NAME} to {EPOCH_B_NAME}")
axes[0].legend(handles=[
    Patch(color="#f0e0c0", label="Desert (unchanged)"),
    Patch(color="#9e9e9e", label="Built-up (unchanged)"),
    Patch(color="#d32f2f", label="Newly built-up"),
], loc="lower right", framealpha=0.9, fontsize=8)

data = [d_lst_s2[m & np.isfinite(d_lst_s2)]
        for m in [stable_desert, stable_builtup, new_builtup]]
axes[1].boxplot(data, labels=["Stable\ndesert", "Stable\nbuilt-up", "Newly\nbuilt-up"],
                showfliers=False)
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("Change in summer LST (degrees C)")
axes[1].set_title("Temperature change by land-use transition")
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.savefig(f"{RESULTS}/03_expansion_heat.png", bbox_inches="tight")
plt.show()

## 7 · Calibrating the lever — how much cooling does albedo actually buy here?

This is the analytical core.

Over built-up pixels only, we regress summer LST on albedo, controlling for
vegetation. The slope tells us **how many degrees of surface temperature one unit
of albedo is worth in this specific city, measured from this city's own data.**

Every cool-roof cost-benefit case we could find imports a sensitivity from
somewhere else. Measuring it locally is cheap, defensible, and it is what turns a
heat map into a decision tool.

**Validation is spatial, not random.** Neighbouring pixels are strongly
autocorrelated — a random train/test split would leak almost-identical pixels
across the boundary and report a flattering, meaningless R². The onboarding guide
flags this exact mistake. We split into ~1 km spatial blocks and hold out whole
blocks.

In [ ]:
built = (lc_b == 2) & np.isfinite(lst_b_s2) & np.isfinite(albedo.values) \
        & np.isfinite(ndvi.values)
print(f"Built-up pixels available for the model: {built.sum():,}")

y = lst_b_s2[built]
X = np.column_stack([albedo.values[built], ndvi.values[built]])

# --- spatial block assignment ------------------------------------------------
BLOCK_PX = int(1000 / RES_S2)        # ~1 km blocks
rr, cc = np.where(built)
block_id = (rr // BLOCK_PX) * 10_000 + (cc // BLOCK_PX)
blocks = np.unique(block_id)
np.random.shuffle(blocks)
K = 5
folds = {b: i % K for i, b in enumerate(blocks)}
fold_of = np.array([folds[b] for b in block_id])
print(f"{len(blocks)} spatial blocks of ~1 km, {K}-fold spatial CV")

oof_pred = np.full_like(y, np.nan, dtype=float)
slopes = []
for k in range(K):
    tr, te = fold_of != k, fold_of == k
    if te.sum() < 30 or tr.sum() < 100:
        continue
    m = LinearRegression().fit(X[tr], y[tr])
    oof_pred[te] = m.predict(X[te])
    slopes.append(m.coef_[0])

ok = np.isfinite(oof_pred)
cv_r2   = r2_score(y[ok], oof_pred[ok])
cv_rmse = float(np.sqrt(mean_squared_error(y[ok], oof_pred[ok])))

final = LinearRegression().fit(X, y)
ALBEDO_SENSITIVITY = float(final.coef_[0])      # degrees C per unit albedo

print("\n--- Albedo to LST model, built-up pixels only ---")
print(f"  LST = {final.intercept_:.2f} "
      f"{final.coef_[0]:+.2f}*albedo {final.coef_[1]:+.2f}*NDVI")
print(f"  in-sample R2        : {final.score(X, y):.3f}")
print(f"  spatial-CV R2       : {cv_r2:.3f}")
print(f"  spatial-CV RMSE     : {cv_rmse:.2f} C")
print(f"  fold slope spread   : {np.min(slopes):.2f} to {np.max(slopes):.2f} "
      f"(sd {np.std(slopes):.2f})")
print(f"\n  MEASURED SENSITIVITY: {ALBEDO_SENSITIVITY:.1f} C per unit albedo")
print(f"  i.e. raising a roof's albedo by 0.10 lowers its summer surface")
print(f"  temperature by about {abs(ALBEDO_SENSITIVITY)*0.10:.2f} C.")

validation_model = dict(cv_r2=cv_r2, cv_rmse=cv_rmse, n=int(built.sum()),
                        n_blocks=int(len(blocks)), folds=K,
                        slope=ALBEDO_SENSITIVITY,
                        slope_sd=float(np.std(slopes)) if slopes else None)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

idx = np.random.choice(len(y), size=min(20000, len(y)), replace=False)
axes[0].scatter(X[idx, 0], y[idx], s=1.5, alpha=0.12, c="#1f3a5f")
xs = np.linspace(np.percentile(X[:,0],1), np.percentile(X[:,0],99), 50)
axes[0].plot(xs, final.intercept_ + final.coef_[0]*xs + final.coef_[1]*np.median(X[:,1]),
             color="#d32f2f", lw=2.2,
             label=f"fit: {final.coef_[0]:.1f} C per unit albedo")
axes[0].set_xlabel("Broadband albedo"); axes[0].set_ylabel("Summer LST (degrees C)")
axes[0].set_title("Darker surfaces run hotter - measured locally")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(oof_pred[ok][:20000], y[ok][:20000], s=1.5, alpha=0.12, c="#1f3a5f")
lo = float(min(np.nanmin(oof_pred[ok]), y[ok].min()))
hi = float(max(np.nanmax(oof_pred[ok]), y[ok].max()))
axes[1].plot([lo, hi], [lo, hi], "k--", lw=1)
axes[1].set_xlabel("Predicted LST, held-out blocks")
axes[1].set_ylabel("Observed LST")
axes[1].set_title(f"Spatial block CV: R2={cv_r2:.2f}, RMSE={cv_rmse:.2f} C")
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f"{RESULTS}/04_albedo_model.png", bbox_inches="tight")
plt.show()

## 8 · From pixels to a worklist — ranking individual rooftops

A heat map is not actionable. A list of addresses is. We pull building footprints
from OpenStreetMap over the focus area, compute each roof's mean albedo and mean
summer LST by zonal statistics, and estimate the cooling each would gain from a
cool-roof retrofit using the sensitivity measured in Section 7.

```
cooling_benefit (C.m2) = roof_area x |sensitivity| x max(0, TARGET_ALBEDO - current_albedo)
```

Area is in the formula deliberately: treating one large warehouse roof is cheaper
per square metre than treating fifty small ones, so a municipality wants its first
contract to cover the biggest, darkest roofs.

In [ ]:
# Overpass rather than osmnx: one HTTP call, no library API churn.
def osm_buildings(bbox, timeout=300):
    w, s, e, n = bbox
    q = f"""
    [out:json][timeout:240];
    (way["building"]({s},{w},{n},{e});
     relation["building"]({s},{w},{n},{e}););
    out geom;
    """
    for url in ["https://overpass-api.de/api/interpreter",
                "https://overpass.kumi.systems/api/interpreter"]:
        try:
            r = requests.post(url, data={"data": q}, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            break
        except Exception as ex:
            print(f"   {url} failed ({type(ex).__name__}), trying next mirror")
    else:
        raise RuntimeError("All Overpass mirrors failed. Retry in a few minutes.")

    polys, props = [], []
    for el in data.get("elements", []):
        geom = el.get("geometry")
        if not geom or len(geom) < 4:
            continue
        try:
            p = Polygon([(g["lon"], g["lat"]) for g in geom])
            if p.is_valid and p.area > 0:
                polys.append(p)
                props.append({"osm_id": el.get("id"),
                              "osm_type": el.get("type"),
                              **{k: v for k, v in (el.get("tags") or {}).items()
                                 if k in ("building", "name", "name:en")}})
        except Exception:
            continue
    gdf = gpd.GeoDataFrame(props, geometry=polys, crs="EPSG:4326")
    return gdf

print("Querying OpenStreetMap (this can take 1-3 minutes)...")
roofs = osm_buildings(FOCUS_BBOX)
print(f"Retrieved {len(roofs):,} building footprints")

roofs = roofs.to_crs(epsg=EPSG)
roofs["area_m2"] = roofs.geometry.area
roofs = roofs[roofs.area_m2 >= 400].reset_index(drop=True)   # >= 2x2 Sentinel pixels
print(f"Kept {len(roofs):,} roofs of at least 400 m2 "
      f"(total {roofs.area_m2.sum()/1e6:.2f} km2 of roof)")

In [ ]:
# Zonal statistics. Both rasters are already on the Sentinel-2 grid and CRS.
alb_arr = albedo.values.astype("float32")
lst_arr = lst_b_s2.astype("float32")
affine  = albedo.rio.transform() if hasattr(albedo, "rio") else None
if affine is None:
    import rioxarray  # noqa
    affine = albedo.rio.transform()

def zmean(arr, name):
    stats = zonal_stats(roofs, arr, affine=affine, stats=["mean"],
                        nodata=np.nan, all_touched=False)
    return pd.Series([s["mean"] if s and s["mean"] is not None else np.nan
                      for s in stats], name=name)

print("Computing zonal statistics per roof...")
roofs["albedo"] = zmean(alb_arr, "albedo").values
roofs["lst_c"]  = zmean(lst_arr, "lst_c").values
roofs = roofs.dropna(subset=["albedo", "lst_c"]).reset_index(drop=True)
print(f"{len(roofs):,} roofs with valid albedo and LST")

# ---- cooling benefit --------------------------------------------------------
sens = abs(ALBEDO_SENSITIVITY)
roofs["albedo_gap"]      = (TARGET_ALBEDO - roofs["albedo"]).clip(lower=0)
roofs["delta_T_est_c"]   = sens * roofs["albedo_gap"]
roofs["cooling_c_m2"]    = roofs["delta_T_est_c"] * roofs["area_m2"]
roofs["lst_anomaly_c"]   = roofs["lst_c"] - roofs["lst_c"].median()

roofs = roofs.sort_values("cooling_c_m2", ascending=False).reset_index(drop=True)
roofs["rank"] = np.arange(1, len(roofs) + 1)

cols = ["rank", "osm_id", "area_m2", "albedo", "lst_c", "lst_anomaly_c",
        "albedo_gap", "delta_T_est_c", "cooling_c_m2"]
out = roofs[cols].round(
    {"area_m2": 0, "albedo": 3, "lst_c": 2, "lst_anomaly_c": 2,
     "albedo_gap": 3, "delta_T_est_c": 2, "cooling_c_m2": 0})
out.to_csv(f"{RESULTS}/roof_priority_ranked.csv", index=False)

print(f"\nWrote {RESULTS}/roof_priority_ranked.csv  ({len(out):,} roofs)")
display(out.head(15))

In [ ]:
top = roofs.head(100)
print(f"--- Top 100 roofs by estimated cooling benefit ---")
print(f"  combined roof area    : {top.area_m2.sum()/1e4:,.1f} hectares "
      f"({top.area_m2.sum()/roofs.area_m2.sum()*100:.1f}% of all roof area)")
print(f"  mean current albedo   : {top.albedo.mean():.3f}")
print(f"  mean current LST      : {top.lst_c.mean():.1f} C")
print(f"  mean estimated cooling: {top.delta_T_est_c.mean():.2f} C per roof")
print(f"  share of total benefit: "
      f"{top.cooling_c_m2.sum()/roofs.cooling_c_m2.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

roofs.plot(ax=axes[0], column="lst_c", cmap="inferno", legend=True,
           legend_kwds={"label": "Mean roof LST (degrees C)", "shrink": 0.7})
axes[0].set_title("Every roof, by surface temperature"); axes[0].set_axis_off()

roofs.plot(ax=axes[1], color="#e8e8e8", edgecolor="none")
top.plot(ax=axes[1], color="#d32f2f", edgecolor="none")
axes[1].set_title("Top 100 priority roofs (red)"); axes[1].set_axis_off()

axes[2].scatter(roofs.albedo, roofs.lst_c, s=np.clip(roofs.area_m2/900, 2, 60),
                alpha=0.35, c="#1f3a5f", edgecolors="none")
axes[2].set_xlabel("Roof albedo"); axes[2].set_ylabel("Roof LST (degrees C)")
axes[2].set_title("Roofs: albedo vs temperature\n(marker size = roof area)")
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f"{RESULTS}/05_roof_priority.png", bbox_inches="tight")
plt.show()

roofs.head(500).to_file(f"{RESULTS}/top500_roofs.geojson", driver="GeoJSON")
print(f"Wrote {RESULTS}/top500_roofs.geojson")

## 9 · Impact, and what a deployed version would look like

The organizers ask how an end user would actually see the results. We do not build
a frontend for the PoC — we state the delivery path and show the artefact it would
serve.

In [ ]:
summary = {
    "aoi": AOI_NAME,
    "aoi_bbox": AOI_BBOX,
    "epochs": {"before": EPOCH_A_NAME, "after": EPOCH_B_NAME},
    "scenes": {"landsat_before": len(items_a), "landsat_after": len(items_b)},
    "mean_lst_before_c": round(float(np.nanmean(lst_a_on_b.values)), 2),
    "mean_lst_after_c": round(float(np.nanmean(lst_b.values)), 2),
    "desert_to_builtup_km2": round(float(new_builtup.sum() * px_km2), 2),
    "albedo_sensitivity_c_per_unit": round(ALBEDO_SENSITIVITY, 2),
    "validation_builtup_classification": {k: (round(v, 4) if isinstance(v, float) else v)
                                          for k, v in validation_builtup.items()},
    "validation_albedo_model": {k: (round(v, 4) if isinstance(v, float) else v)
                                for k, v in validation_model.items()},
    "roofs_analysed": int(len(roofs)),
    "roof_area_total_km2": round(float(roofs.area_m2.sum() / 1e6), 3),
    "top100_cooling_share_pct": round(float(
        roofs.head(100).cooling_c_m2.sum() / roofs.cooling_c_m2.sum() * 100), 1),
}
with open(f"{RESULTS}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

### How this reaches a decision-maker

Not built for the PoC, but this is the path:

1. **Quarterly screening report (PDF/CSV).** The pipeline reruns each summer and
   emails the municipality's asset team a refreshed top-200 worklist. This is the
   lowest-friction entry point — it needs no new software on their side.
2. **GIS layer.** `top500_roofs.geojson` drops straight into the municipality's
   existing ArcGIS/QGIS environment next to their cadastral and permit layers.
3. **API.** A `/priority?district=` endpoint returning ranked roofs, so the
   ranking can be read by a works-planning system.
4. **Verification loop.** After a retrofit, rerun on the next summer's imagery and
   measure the realised ΔT against the predicted one — turning the tool into a
   contract-compliance check, not just a planning aid.

### Limitations — state these plainly, they are scored under *validity*

- **LST is not air temperature.** Surface temperature is what satellites measure.
  Human thermal comfort depends on air temperature, humidity and radiant load.
  Our output ranks surfaces, it does not predict heat-stroke risk.
- **The sensitivity is correlational.** Section 7 measures how albedo and LST
  covary across space; it is not a controlled experiment. Confounders (roof
  material thermal mass, building use, air-conditioning exhaust) are not separated.
- **Validation is against a model, not ground truth.** ESA WorldCover is itself a
  classification product. No field survey was conducted.
- **30 m thermal pixels over smaller roofs are mixed.** Roofs below ~900 m² carry
  signal from neighbouring surfaces. We keep roofs ≥400 m² but the smallest are the
  least reliable; treat the large-roof end of the ranking as the trustworthy part.
- **OSM completeness varies.** Missing footprints are missing from the ranking.
- **Landsat overpass is ~10:30 local.** Peak surface temperature in Abu Dhabi is
  later in the afternoon, so absolute values understate the daily maximum.


## 10 · Optional (Tier 2) — Tanager hyperspectral roof material screening

**Run only if `00_recon.ipynb` found a Tanager `urban` scene over the UAE.**

Albedo tells us how bright a roof is. Hyperspectral tells us what it is *made of* —
and material determines which retrofit is appropriate and what it will cost. With
426 narrow bands we can separate metal roofing from concrete from bitumen, which
six broad Sentinel-2 bands cannot do.

If no scene covers the AOI, **do not fake this.** Delete the section and say so in
the README. A clean negative is worth more than a fabricated positive.

In [ ]:
RUN_TIER2 = False      # set True only if recon found a UAE urban scene
TANAGER_ID = ""        # paste the scene ID from 00_recon.ipynb

if RUN_TIER2 and TANAGER_ID:
    import h5py
    base = "https://www.planet.com/data/stac/tanager-core-imagery/urban"
    item = requests.get(f"{base}/{TANAGER_ID}/{TANAGER_ID}.json", timeout=60).json()
    sr_key = "ortho_sr_hdf5" if "ortho_sr_hdf5" in item["assets"] else "basic_sr_hdf5"
    sr_url = item["assets"][sr_key]["href"]
    local = os.path.join(SAMPLE, os.path.basename(sr_url))
    if not os.path.exists(local):
        print("Downloading Tanager cube (several minutes)...")
        with requests.get(sr_url, stream=True) as r:
            r.raise_for_status()
            with open(local, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    fh.write(chunk)
    print("cube:", local)

    bands_meta = item["assets"][sr_key].get("bands", [])
    wl = np.array([b["eo:center_wavelength"] for b in bands_meta
                   if "eo:center_wavelength" in b]) * 1000.0

    def bad_band(w):     # water-vapour troughs and the noisy SWIR edge
        return (1350 <= w <= 1450) or (1800 <= w <= 1950) or w > 2450
    good = [i for i, w in enumerate(wl) if not bad_band(w)]

    root = "HDFEOS/GRIDS/HYP/Data Fields"
    with h5py.File(local, "r") as f:
        cloud  = f[f"{root}/beta_cloud_mask"][:]
        nodata = f[f"{root}/nodata_pixels"][:]
        validp = (cloud == 0) & (nodata == 0)
        cube = f[f"{root}/surface_reflectance"][good, :, :].astype("float32")
    cube[:, ~validp] = np.nan
    cube[cube < 0] = np.nan
    print("cube shape (bands, y, x):", cube.shape)

    # Spectral Angle Mapper against endmembers taken from the scene itself.
    # Replace these row/col pairs with pixels you have visually confirmed.
    ENDMEMBERS = {
        "bright_roof":   (100, 100),
        "dark_bitumen":  (150, 200),
        "concrete":      (200, 150),
        "bare_sand":     (300, 300),
    }
    def sam(cube, ref):
        num = np.nansum(cube * ref[:, None, None], axis=0)
        den = np.sqrt(np.nansum(cube**2, axis=0)) * np.sqrt(np.nansum(ref**2))
        return np.arccos(np.clip(num / (den + 1e-9), -1, 1))

    angles, names = [], []
    for name, (r, c) in ENDMEMBERS.items():
        ref = cube[:, r, c]
        if not np.isfinite(ref).any():
            print(f"  endmember {name} at ({r},{c}) is invalid - pick another pixel")
            continue
        angles.append(sam(cube, ref)); names.append(name)

    if angles:
        material = np.argmin(np.stack(angles), axis=0).astype("float32")
        material[~validp] = np.nan
        plt.figure(figsize=(9, 7))
        plt.imshow(material, cmap="tab10"); plt.axis("off")
        plt.title("Tanager hyperspectral - roof material screening (SAM)")
        plt.colorbar(ticks=range(len(names)), format=plt.FuncFormatter(
            lambda v, p: names[int(v)] if 0 <= int(v) < len(names) else ""))
        plt.savefig(f"{RESULTS}/06_material_sam.png", bbox_inches="tight")
        plt.show()
else:
    print("Tier 2 skipped. Set RUN_TIER2=True and paste a scene ID to enable.")

## 11 · Save a small sample input so a reviewer can reproduce this

The submission guide requires committed example input. Full scenes are far too
large for GitHub, so we save a clipped GeoTIFF of the focus area plus the exact
query parameters that produced it.

In [ ]:
import rioxarray  # noqa

clip = dict(x=slice(None), y=slice(None))
try:
    alb_small = albedo.rio.write_crs(EPSG).rio.clip_box(
        *gpd.GeoSeries([box(*FOCUS_BBOX)], crs=4326).to_crs(EPSG).total_bounds)
    alb_small.rio.to_raster(f"{SAMPLE}/albedo_focus.tif", compress="deflate")
    print(f"wrote {SAMPLE}/albedo_focus.tif  "
          f"({os.path.getsize(f'{SAMPLE}/albedo_focus.tif')/1e6:.1f} MB)")
except Exception as e:
    print("clip failed:", e)

provenance = {
    "generated_utc": pd.Timestamp.utcnow().isoformat(),
    "aoi_bbox": AOI_BBOX, "focus_bbox": FOCUS_BBOX,
    "epsg": EPSG,
    "landsat": {"collection": "landsat-c2-l2", "windows": [EPOCH_A, EPOCH_B],
                "max_cloud_pct": MAX_CLOUD,
                "scene_ids_before": [i.id for i in items_a][:50],
                "scene_ids_after":  [i.id for i in items_b][:50]},
    "sentinel2": {"collection": "sentinel-2-l2a", "window": EPOCH_B,
                  "bands": S2_BANDS, "max_cloud_pct": MAX_CLOUD},
    "worldcover": {"collection": "esa-worldcover", "builtup_class": 50},
    "osm": {"source": "Overpass API", "bbox": FOCUS_BBOX,
            "min_roof_area_m2": 400},
    "licences": {
        "Landsat": "Public domain (USGS)",
        "Sentinel-2": "Copernicus Sentinel data, free and open",
        "ESA WorldCover": "CC BY 4.0",
        "OpenStreetMap": "ODbL - (c) OpenStreetMap contributors",
        "Planet Tanager open archive": "CC BY 4.0 (c) Planet Labs PBC",
    },
}
with open(f"{SAMPLE}/provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)
print(f"wrote {SAMPLE}/provenance.json")

print("\nFiles produced:")
for d in (RESULTS, SAMPLE):
    for fn in sorted(os.listdir(d)):
        p = os.path.join(d, fn)
        print(f"   {p:50s} {os.path.getsize(p)/1e3:9.1f} KB")

---

## Done

Numbers to carry into the README and the slide deck:

- scenes used per epoch, and the date ranges
- mean LST before and after, and the desert-to-built-up conversion in km²
- **the headline:** how much more newly-built land warmed than untouched desert
- built-up classification accuracy / precision / recall / F1 / IoU vs WorldCover
- **the measured albedo sensitivity**, with spatial-CV R² and RMSE
- roofs analysed, and the share of total benefit concentrated in the top 100

All of it is in `results/summary.json`.
